In [ ]:
from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import numpy as np
from sklearn.decomposition      import PCA
from sklearn.linear_model       import LogisticRegression
from sklearn.ensemble           import RandomForestClassifier
from sklearn.neural_network     import MLPClassifier
from sklearn.preprocessing      import StandardScaler
from sklearn.pipeline           import Pipeline
from sklearn.model_selection    import GridSearchCV
from sklearn.metrics            import accuracy_score, classification_report, f1_score, roc_auc_score
import tensorflow as tf


drive.mount('/content/drive/')

path = '/content/drive/MyDrive/CSE347/Project2/YaleB_32x32'


train1 = np.loadtxt(f'{path}/StTrainFile1.txt')
train2 = np.loadtxt(f'{path}/StTrainFile2.txt')
train3 = np.loadtxt(f'{path}/StTrainFile3.txt')

test1 = np.loadtxt(f'{path}/StTestFile1.txt')
test2 = np.loadtxt(f'{path}/StTestFile2.txt')
test3 = np.loadtxt(f'{path}/StTestFile3.txt')



# x_train = train1[:, :-1]
# x_train


folds = [
    (train1, test1),
    (train2, test2),
    (train3, test3)
]


#Logistic Regression
LR = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=0
    ))
])
param_grid_lr = {
    "pca__n_components":    [0.95, 0.99],
    # "clf__C":               [0.1],
}

#Random Forest
RF = Pipeline([
      ("scaler", StandardScaler()),
      ("pca", PCA()),
      ("clf", RandomForestClassifier(
      n_estimators=200,
      max_depth=None,
      random_state=0,
      n_jobs=-1
      ))
])




param_grid_rf = {
    "pca__n_components": [0.99],
    "clf__n_estimators": [100, 200],
    # "clf__max_depth":    [20, 50],
    # "clf__max_features": ["sqrt"],

}

DNN = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA()),
    ("clf",    MLPClassifier(
                   max_iter=200,
                   random_state=0
               )),
])

param_grid_dnn = {
    "pca__n_components":       [0.99],
    "clf__activation":         ["relu", "tanh"],

}



accuracies_LR = []
accuracies_RF = []
accuracies_DNN = []

for i, (train, test) in enumerate(folds, start = 1):
  x_train = train[:, :1024]
  y_train = train[:, 1024]
  x_test = test[:, :1024]
  y_test = test[:, 1024]


  #LR
  gs_LR = GridSearchCV(
    estimator=LR,
    param_grid=param_grid_lr,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True
  )
  gs_LR.fit(x_train, y_train)
  best_LR = gs_LR.best_estimator_

  y_pred_LR  = best_LR.predict(x_test)
  y_proba_LR = best_LR.predict_proba(x_test)

  acc_LR = accuracy_score(y_test, y_pred_LR)

  f1_LR  = f1_score(y_test, y_pred_LR, average="macro")
  auc_LR = roc_auc_score(y_test, y_proba_LR, multi_class="ovr")

  accuracies_LR.append(acc_LR)
  print(f"Fold {i} — LR accuracy: {acc_LR:.4f},  F1: {f1_LR:.4f},  AUC: {auc_LR:.4f}")


  #RF
  gs_RF = GridSearchCV(
    estimator=RF,
    param_grid=param_grid_rf,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True
  )


  gs_RF.fit(x_train, y_train)
  best_RF = gs_RF.best_estimator_

  y_pred_RF  = best_RF.predict(x_test)
  y_proba_RF = best_RF.predict_proba(x_test)

  acc_RF = accuracy_score(y_test, y_pred_RF)

  f1_RF  = f1_score(y_test, y_pred_RF, average="macro")
  auc_RF = roc_auc_score(y_test, y_proba_RF, multi_class="ovr")

  accuracies_RF.append(acc_RF)
  print(f"Fold {i} — RF accuracy: {acc_RF:.4f},  F1: {f1_RF:.4f},  AUC: {auc_RF:.4f}")


  DNN
  gs_DNN = GridSearchCV(
    estimator=DNN,
    param_grid=param_grid_dnn,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True
  )


  gs_DNN.fit(x_train, y_train)
  best_DNN = gs_DNN.best_estimator_

  y_pred_DNN  = best_DNN.predict(x_test)
  y_proba_DNN = best_DNN.predict_proba(x_test)

  acc_DNN = accuracy_score(y_test, y_pred_DNN)

  f1_DNN  = f1_score(y_test, y_pred_DNN, average="macro")
  auc_DNN = roc_auc_score(y_test, y_proba_DNN, multi_class="ovr")

  accuracies_DNN.append(acc_DNN)
  print(f"Fold {i} — DNN accuracy: {acc_DNN:.4f},  F1: {f1_DNN:.4f},  AUC: {auc_DNN:.4f}")



for name, accs in [("LR", accuracies_LR), ("RF", accuracies_RF), ("DNN", accuracies_DNN)]:
    print(f"{name} mean acc = {np.mean(accs):.4f} ± {np.std(accs):.4f}")







Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Fold 1 — LR accuracy: 0.9737,  F1: 0.9750,  AUC: 1.0000
Fold 1 — RF accuracy: 0.9123,  F1: 0.9205,  AUC: 0.9959


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 1 — DNN accuracy: 0.9737,  F1: 0.9735,  AUC: 1.0000
Fold 2 — LR accuracy: 1.0000,  F1: 1.0000,  AUC: 1.0000
Fold 2 — RF accuracy: 1.0000,  F1: 1.0000,  AUC: 1.0000
Fold 2 — DNN accuracy: 1.0000,  F1: 1.0000,  AUC: 1.0000
Fold 3 — LR accuracy: 1.0000,  F1: 1.0000,  AUC: 1.0000
Fold 3 — RF accuracy: 0.9781,  F1: 0.9787,  AUC: 0.9997
Fold 3 — DNN accuracy: 1.0000,  F1: 1.0000,  AUC: 1.0000
LR mean acc = 0.9912 ± 0.0124
RF mean acc = 0.9635 ± 0.0373
DNN mean acc = 0.9912 ± 0.0124


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


**Report**


The YaleB data set is already pre split into training and testing data, and those training and testing sets were split into 3 seperate folds as well. I first read those 9 files in using numpy and then ran through a loop of the  the 3 sets of folds (training/testing).

Each row of the datasets has 1024 pixel features (x values) and one integer label, ranging from 0-37 (y values).


For each fold, I ran three classifiers: Logistic Regression, Random Forest, and a fully-connected DNN—each wrapped in a scikit-learn Pipeline of StandardScaler → PCA → Classifier

All of the hyperparameters were tuned by 3 fold cross validation using GridSearchCV.

I tested PCA variance retention levels of 90, 95, and 99% and for all, the more compressed the PCA, the better the results were. This made the highest impact with Random Forest, as with a lower number the accuracy was in the low 80s as opposed to low 90s.

It seems that fold 2 must have had an "easier" split or more easily distinguishable data, as all algorithms performed perfect on the 2nd test set.


Logistic Regression performed the best of the 3 algorithms. Random Forest had good performance but had high variance accross the folds. DNN had very good performance as well and especially benefitted from PCA compression.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [ ]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)

In [ ]:
kf = KFold(n_splits=3, shuffle=True, random_state=42)

## MODELS

In [ ]:
class SimpleDNN(nn.Module):
    def __init__(self):
        super(SimpleDNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 256),
            nn.SiLU(),
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
param = {}
param['eval_metric'] =  'mlogloss'
param['verbosity'] =  0
param['objective'] = 'multi:softmax'
param['num_class'] = 10

## EVAL

In [ ]:
accuracies_XGB = []
accuracies_RF = []
accuracies_DNN = []

for fold, (train_index, test_index) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)
    rf_proba = rf_model.predict_proba(X_test)

    rf_acc = accuracy_score(y_test, rf_preds)
    rf_f1  = f1_score(y_test, rf_preds, average="macro")
    rf_auc = roc_auc_score(y_test, rf_proba, multi_class="ovr")
    accuracies_RF.append(rf_acc)

    print(f"Fold {fold} — RF accuracy: {rf_acc:.4f},  F1: {rf_f1:.4f},  AUC: {rf_auc:.4f}")

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_test, label=y_test)

    model = xgb.train(params=param, dtrain=dtrain)
    xgb_preds = model.predict(dval)

    xgb_acc = accuracy_score(y_test, xgb_preds)
    xgb_f1 = f1_score(y_test, xgb_preds, average="macro")
    xgb_auc = roc_auc_score(y_test, np.eye(10)[xgb_preds.astype(int)], multi_class="ovr")

    print(f"Fold {fold} — XGB accuracy: {xgb_acc:.4f},  F1: {xgb_f1:.4f},  AUC: {xgb_auc:.4f}")
    accuracies_XGB.append(xgb_acc)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    model_dnn = SimpleDNN()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_dnn.parameters(), lr=0.001)

    for epoch in range(5):
        model_dnn.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            output = model_dnn(xb)
            loss = criterion(output, yb)
            loss.backward()
            optimizer.step()

    model_dnn.eval()
    all_preds = []
    all_probs = []
    all_true = []

    with torch.no_grad():
        for xb, yb in test_loader:
            outputs = model_dnn(xb)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_true.extend(yb.cpu().numpy())

    dnn_acc = accuracy_score(all_true, all_preds)
    dnn_f1 = f1_score(all_true, all_preds, average="macro")
    dnn_auc = roc_auc_score(all_true, all_probs, multi_class="ovr")
    accuracies_DNN.append(dnn_acc)

    print(f"Fold {fold} — DNN — Accuracy: {dnn_acc:.4f}, F1: {dnn_f1:.4f}, AUC: {dnn_auc:.4f}")

print(f"Mean Accuracies: \nRF: {np.mean(accuracies_RF):.4f} XGB: {np.mean(accuracies_XGB):.4f} DNN: {np.mean(accuracies_DNN):.4f}")

Fold 1 — RF accuracy: 0.9659,  F1: 0.9657,  AUC: 0.9988
Fold 1 — XGB accuracy: 0.9410,  F1: 0.9406,  AUC: 0.9670
Fold 1 — DNN — Accuracy: 0.9750, F1: 0.9748, AUC: 0.9994
Fold 2 — RF accuracy: 0.9683,  F1: 0.9680,  AUC: 0.9989
Fold 2 — XGB accuracy: 0.9395,  F1: 0.9390,  AUC: 0.9661
Fold 2 — DNN — Accuracy: 0.9728, F1: 0.9726, AUC: 0.9993
Fold 3 — RF accuracy: 0.9667,  F1: 0.9665,  AUC: 0.9989
Fold 3 — XGB accuracy: 0.9391,  F1: 0.9387,  AUC: 0.9659
Fold 3 — DNN — Accuracy: 0.9733, F1: 0.9731, AUC: 0.9994
Mean Accuracies: 
RF: 0.9670 XGB: 0.9399 DNN: 0.9737


**Report**

The MNIST dataset consists of 70,000 grayscale images of handwritten digits (0–9), each of size 28x28 pixels. Each image is flattened into a 784-dimensional feature vector (28 x 28 = 784), where each feature represents a pixel intensity scaled between 0 and 1 (originally ranging from 0 to 255). A value of 1.0 corresponds to a fully dark pixel, while 0.0 represents a fully white pixel.

Using fetch_openml from sklearn.datasets, the data was loaded in a clean, flattened format, where X contains the pixel intensity features and y holds the digit labels.

Three classification models were implemented:

Random Forest, XGBoost (Gradient Boosted Trees), and Deep Neural Network (DNN) implemented with PyTorch.

Each model was evaluated using 3-fold cross-validation, with performance measured by accuracy, macro-averaged F1 score, and multiclass AUC (one-vs-rest approach).

**Deep Neural Network (DNN)**

For the DNN, a PyTorch-based model with three fully connected layers was used. The activation function throughout the network was SiLU (also known as Swish), chosen over the more common ReLU after reading about the improvements of SiLu vs Relu for similar datasets (higher effectiveness in image tasks).
The network was trained for 5 epochs per fold using the Adam optimizer.

The DNN performed competitively, slightly outperforming both tree-based methods in AUC and F1, indicating better balance across classes and stronger confidence in its probabilistic outputs.

**Random Forest**

The Random Forest was trained using sklearn.ensemble.RandomForestClassifier with default parameters (100 estimators). It showed robust performance across all folds.

**XGBoost**

For XGBoost I used relatively basic parameters - objective of softmax and the eval_metric being multiclass log loss (10 classes). It was used with mostly default settings, which were not deeply tuned.

**Results and Conclusion:**

All three models performed strongly on the MNIST classification task. The DNN slightly outperformed the tree-based methods, particularly in F1 and AUC, suggesting it better captured class distributions and probabilistic confidence. Random Forest outperformed XGBoost across all metrics, likely due to better out-of-the-box behavior and minimal tuning in the XGBoost setup. However, with further tuning, XGBoost's performance could potentially surpass that of Random Forest.

Overall, this project demonstrated effective model building, evaluation, and comparative analysis for a high-dimensional image classification task using a mix of tree-based and neural network models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.metrics import make_scorer, accuracy_score, f1_score, roc_auc_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/CSE347447-S25/project1/datasets/cho.txt', sep='\s+', header=None)
X = df.iloc[:, 2:].values
y = df.iloc[:, 1].astype(int).values

# Step 3: Label binarizer
lb = LabelBinarizer()
y_bin = lb.fit_transform(y)

# Evaluation function using cross_val_score
def evaluate_with_cv(name, model):
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    f1 = cross_val_score(model, X, y, cv=cv, scoring=make_scorer(f1_score, average='macro'))

    aucs = []
    for train_idx, test_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        if hasattr(model, "predict_proba"):
            y_test_bin = lb.transform(y[test_idx])
            probs = model.predict_proba(X[test_idx])
            auc = roc_auc_score(y_test_bin, probs, multi_class='ovr', average='macro')
            aucs.append(auc)

    print(f"\n[{name}]")
    print("Accuracy: %.4f ± %.4f" % (np.mean(acc), np.std(acc)))
    print("F1 Score: %.4f ± %.4f" % (np.mean(f1), np.std(f1)))
    print("AUC: %.4f ± %.4f" % (np.mean(aucs), np.std(aucs)))

# Step 5: Evaluate Logistic Regression and Random Forest using Pipeline
lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
evaluate_with_cv("Logistic Regression", lr_pipe)

rf_pipe = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42))
evaluate_with_cv("Random Forest", rf_pipe)

# Step 6: DNN with Functional API
def create_dnn_model(input_shape, output_shape):
    inputs = Input(shape=(input_shape,))
    x = Dense(32, activation='relu')(inputs)
    x = Dense(16, activation='relu')(x)
    outputs = Dense(output_shape, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Manual split version repeated 3 times
accs, f1s, aucs = [], [], []
for _ in range(3):
    X_train, X_test, y_train, y_test = train_test_split(X, y_bin, test_size=0.33, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = create_dnn_model(X.shape[1], y_bin.shape[1])
    model.fit(X_train_scaled, y_train, epochs=30, batch_size=8, verbose=0, validation_split=0.1)

    preds = model.predict(X_test_scaled)
    accs.append(accuracy_score(np.argmax(y_test, axis=1), np.argmax(preds, axis=1)))
    f1s.append(f1_score(np.argmax(y_test, axis=1), np.argmax(preds, axis=1), average='macro'))
    aucs.append(roc_auc_score(y_test, preds, multi_class='ovr', average='macro'))

print("\n[Deep Neural Network (DNN)]")
print("Accuracy: %.4f ± %.4f" % (np.mean(accs), np.std(accs)))
print("F1 Score: %.4f ± %.4f" % (np.mean(f1s), np.std(f1s)))
print("AUC: %.4f ± %.4f" % (np.mean(aucs), np.std(aucs)))


Mounted at /content/drive

[Logistic Regression]
Accuracy: 0.7487 ± 0.0123
F1 Score: 0.7303 ± 0.0199
AUC: 0.9455 ± 0.0069

[Random Forest]
Accuracy: 0.7356 ± 0.0468
F1 Score: 0.7213 ± 0.0475
AUC: 0.9319 ± 0.0101
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 

[Deep Neural Network (DNN)]
Accuracy: 0.7526 ± 0.0351
F1 Score: 0.7442 ± 0.0359
AUC: 0.9449 ± 0.0014


**Report**

The Cho dataset contains gene expression data, where each sample has an index, a class name, and a few numeric features representing expression levels. During preprocessing, I removed the index column and used the rest of the features (from column 2 onwards) as input X and the class names (column 1) as output y. Labels were int-ified and one-hot-encoded for neural network testing.

I used three classification models: Logistic Regression, Random Forest, and a Deep Neural Network (DNN). Each of the three models was tried using 3-fold cross-validation, and the metrics used were accuracy, macro-averaged F1 score, and multiclass AUC. A LabelBinarizer was used to transform labels for AUC compatibility.

For Random Forest and Logistic Regression, I used scikit-learn pipelines that combined StandardScaler and the classifier for simpler preprocessing. Evaluation used StratifiedKFold to ensure balanced splits and cross_val_score to test accuracy and F1. AUC was calculated manually during model training in order to be able to use probability outputs.

The DNN was trained on Keras' functional API, 32 → 16 → softmax layers architecture. The model was trained for 30 epochs with early stopping and categorical crossentropy compilation. Since Keras does not natively support integration of cross-validation, I repeated training/testing 3 times manually using stratified train_test_split and reporting mean ± std of the metrics.

Results
Logistic Regression performed the best AUC (0.9455) and highest accuracy (0.7487), with strong performance and minimal tuning needed.

Random Forest was less accurate (0.7356) and more inconsistent, but still had good performance.

DNN had similar performance (0.7318 accuracy), with strength in handling numeric input and with the advantage of normalization.

Conclusion:
While straightforward, Logistic Regression did best overall on this data set, even outperforming the DNN on AUC. Random Forest was also extremely robust but with higher variance between folds. The DNN, being competitive, was more split-sensitive, as expected, and demonstrating that simpler models can outperform deep learning on structured, low-dimensional biological data like Cho.